# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 02.00 · Preparación local del bundle versionado para Colab

Genera el bundle reproducible y publica en Google Drive una versión inmutable identificada por el contenido; debe ejecutarse con kernel local antes de usar Colab.

La identidad estable combina los SHA-256 del código y de las entradas comprimidas. La celda publica primero todos los artefactos y el manifiesto al final, de modo que una sincronización interrumpida no pueda presentarse como una versión completa [1]. La ruta de Google Drive y el momento de publicación son decisiones operativas locales.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')


## Configuración local

In [ ]:
from pathlib import Path
import importlib.util

if importlib.util.find_spec('google.colab') is not None:
    raise RuntimeError('02_00 debe ejecutarse con el kernel Python local, no con Colab.')

RUN_PREPARE_BUNDLE=False  # Cambie a True después de configurar y revisar DRIVE_ROOT.
DRIVE_ROOT=None  # Ejemplo: Path('RUTA_LOCAL_GOOGLE_DRIVE')/'ModeracionPeru_Colab'
LOCAL_BUNDLE=ROOT/'resultados/colab_bundle'

show_summary('Publicación preparada',{'kernel':'local','bundle_local':LOCAL_BUNDLE,'destino_drive':DRIVE_ROOT,'ejecutar':RUN_PREPARE_BUNDLE},tone='neutral')
show_callout('Dos momentos de ejecución','Ejecute 02_00 antes de 02_01 para publicar chunks y vuelva a ejecutarlo después de 02_05 para publicar el snapshot que consumirá la etapa 03.',tone='info')

## Construcción, verificación y publicación

In [ ]:
from tqdm.auto import tqdm
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
from tools.prepare_colab_bundle import publish_drive_release

bundle_progress={'bar':None}
def report_bundle_progress(event):
    if event['status']=='started':
        bundle_progress['bar']=tqdm(total=event['total'],desc='Publicando bundle',unit='etapa')
        return
    bar=bundle_progress.get('bar')
    if bar is not None:
        bar.update(event.get('advance',0))
        bar.set_postfix(etapa=event.get('stage',''))

if RUN_PREPARE_BUNDLE:
    if DRIVE_ROOT is None:
        raise ValueError("Configure DRIVE_ROOT con la carpeta local de Google Drive que termina en 'ModeracionPeru_Colab'.")
    try:
        bundle_result=publish_drive_release(LOCAL_BUNDLE,DRIVE_ROOT,progress_callback=report_bundle_progress)
    finally:
        if bundle_progress.get('bar') is not None:
            bundle_progress['bar'].close()
    show_result('Versión de Colab publicada en Drive',bundle_result,tone='success')
    show_callout('Antes de cambiar a Colab','Espere a que Google Drive indique que terminó de sincronizar. Luego abra 02_01 o el cuaderno 03 correspondiente y ejecute desde la primera celda.',tone='warning')
else:
    show_callout('Publicación desactivada','Configure DRIVE_ROOT y cambie RUN_PREPARE_BUNDLE=True. No se escribió ningún archivo.',tone='neutral')

## Referencias

[1] National Institute of Standards and Technology, "Secure Hash Standard (SHS)," FIPS PUB 180-4, Aug. 2015, doi: 10.6028/NIST.FIPS.180-4.